# AI for Handwriting Identification — Colab quickstart

**Colab has no `C:\Users\...`.** On your PC the release may live at `C:\Users\thisb\Downloads\AnyScriptFiltered.tar.gz`. For Colab: **upload that `.tar.gz` to Google Drive** (e.g. `MyDrive/`), then use a **`/content/drive/MyDrive/...`** path below and extract there so training survives disconnects.

**Dataset:** `DATA_ROOT` = folder whose **children are author IDs** (default `/content/drive/MyDrive/AnyScriptFiltered`). Layouts: `author/book/*.jpg` or `author/*.png`.

## Part A, B, C (run via `/content/ai-hw/scripts/...`; shell stays in `/content` so `pip` never breaks after `rm -rf`)

| Part | Script | Output |
|------|--------|--------|
| **A — Train** | `train_triplet_unsloth.py` | `OUT/best.pt` |
| **B — FAISS** | `build_faiss_index.py` | `OUT/faiss.index`, `OUT/meta.npy` |
| **C — Submit CSV** | `export_anyscript_submission.py` | dense `intra_book` / `extra_book` CSV |

More detail: **`README.md`** (Quick start + ICDAR submission).

In [ ]:
# Optional: persist data & checkpoints on Drive
from google.colab import drive
drive.mount("/content/drive")

### Reset working directory (run if you see `getcwd` / `Unable to read current working directory`)

If your shell was inside `/content/ai-hw` and that folder was deleted, **run the next cell** before **Clone** or **`git pull`**.

In [ ]:
# Clone repo (fork/URL ok — change if you renamed the repo)
# Do NOT add %cd /content/ai-hw/scripts — re-run + rm deletes cwd (getcwd / git clone fail).
%cd /content
import os
os.chdir("/content")
!rm -rf /content/ai-hw
!git clone https://github.com/Todd838/AI-for-Handwriting-Identification.git /content/ai-hw
# Do not %cd into the repo: if you re-run this cell, rm -rf removes your cwd and pip breaks.
print("Repo:", "/content/ai-hw/scripts — run scripts with full path below.")

In [ ]:
# Install everything from the repo (same as local requirements.txt)
import os
os.chdir("/content")
%pip install -q -r /content/ai-hw/requirements.txt
# If pip errors on a package, run: %pip install -q unsloth  (after GPU runtime is on)

In [ ]:
# Unsloth (use Runtime → GPU first). If install fails, train with --no_unsloth
import os
os.chdir("/content")
import torch
print("CUDA available:", torch.cuda.is_available())
%pip install -q unsloth

### GPU required for Unsloth

**`CUDA available: False`** means this runtime has **no GPU**. Use **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (or L4 / A100 if offered), then **Runtime → Restart runtime** and **re-run** cells from the top (mount → clone → install → this cell).

Without a GPU you can still try training with **`--no_unsloth`** (much slower / may OOM on large models).

In [ ]:
# Dataset on Colab = Drive path, NOT C:\Users\thisb\Downloads\...
# Upload AnyScriptFiltered.tar.gz to Drive, then set ARCHIVE. After extract, we auto-pick
# DATA_ROOT (tar often creates MyDrive/data/datasets/.../binarized, not MyDrive/AnyScriptFiltered).
import os
import subprocess
import sys
os.chdir("/content")

EXTRACT_PARENT = "/content/drive/MyDrive"

sys.path.insert(0, "/content/ai-hw/scripts")
from data_anyscript import (
    colab_anyscript_archive_candidates,
    colab_drive_data_root_candidates,
    resolve_colab_data_root_any,
)

CANDIDATES = colab_drive_data_root_candidates(EXTRACT_PARENT)
DATA_ROOT = CANDIDATES[0]

ARCHIVE = None
for ap in colab_anyscript_archive_candidates(EXTRACT_PARENT):
    if os.path.isfile(ap):
        ARCHIVE = ap
        break
if ARCHIVE:
    print("Extracting from", ARCHIVE, "(long run; needs Drive space)...")
    os.makedirs(EXTRACT_PARENT, exist_ok=True)
    subprocess.run(["tar", "-xzf", ARCHIVE, "-C", EXTRACT_PARENT], check=False)
else:
    print("No archive at:", colab_anyscript_archive_candidates(EXTRACT_PARENT))

picked = resolve_colab_data_root_any()
if picked:
    DATA_ROOT = picked
    print("OK: DATA_ROOT (auto) ->", DATA_ROOT)
else:
    print("WARNING: no folder with >=2 authors and 2+ pages each. Tried:")
    for c in CANDIDATES:
        print(" ", c, "exists=" + str(os.path.isdir(c)))
    print("Upload/extract data, or run inspect_anyscript_layout.py (next section).")
    DATA_ROOT = CANDIDATES[0]

## Part A — training

Re-run the **dataset** cell above **in this session** before training so `DATA_ROOT` is set (it must not stay the empty `AnyScriptFiltered` folder). Uncomment the `!python` line. Checkpoints go to **`OUT`** on Drive.

### If Part A says `0 pages found`

1. If `AnyScriptFiltered/` only has **`.tar.gz`**, you must **extract** first (the dataset cell now tries both `MyDrive/AnyScriptFiltered.tar.gz` and `MyDrive/AnyScriptFiltered/AnyScriptFiltered.tar.gz`).
2. Use **`--data_root auto`** in the train command so path resolution runs inside `train_triplet_unsloth.py`.
3. **Do not** use `DATA_ROOT` from unrelated projects (e.g. other Kaggle folders). The inspector only suggests **AnyScript-related** paths unless you pass `--force-unrelated-scan`.

In [ ]:
# After clone: finds which subfolder actually contains page images.
%cd /content
!python /content/ai-hw/scripts/inspect_anyscript_layout.py /content/drive/MyDrive/AnyScriptFiltered

In [ ]:
OUT = "/content/drive/MyDrive/anyscript_runs/run1"
# import os; os.chdir("/content")
# Use --data_root auto so training searches Drive in-process (avoids stale {DATA_ROOT}).
# Or pass {DATA_ROOT} if you set it manually in the dataset cell.
# !python /content/ai-hw/scripts/train_triplet_unsloth.py --data_root auto --model_name THUDM/glm-ocr --output_dir {OUT} --epochs 1 --steps_per_epoch 200 --batch_size 2
# Add --no_unsloth if Unsloth fails

## Part B — FAISS index

Run after **`best.pt`** exists under `OUT`. Uncomment the next cell.

In [ ]:
# Part B streams images from disk in batches (safe for huge galleries). Lower --batch_size if GPU OOM.
# import os; os.chdir("/content")
# !python /content/ai-hw/scripts/build_faiss_index.py --data_root {DATA_ROOT} --checkpoint {OUT}/best.pt --index_out {OUT}/faiss.index --meta_out {OUT}/meta.npy --batch_size 4
# Add --no_unsloth if needed (match Part A).

## Part C — submission CSV

Set **`QUERY_ROOT`** to held-out query images. **`GALLERY_ROOT`** is usually **`DATA_ROOT`**. For a real upload, add **`--query_ids_json`** and **`--gallery_ids_json`** (see **`README.md`**). **`--allow_synthetic_ids`** is only for testing.

In [ ]:
# QUERY_ROOT = "/content/drive/MyDrive/anyscript_query_holdout"
# GALLERY_ROOT = DATA_ROOT
# import os; os.chdir("/content")
# !python /content/ai-hw/scripts/export_anyscript_submission.py --mode intra_book --checkpoint {OUT}/best.pt --gallery_data_root {GALLERY_ROOT} --query_data_root {QUERY_ROOT} --out_csv {OUT}/submission_intra_book.csv --batch_size 4 --allow_synthetic_ids
# !python /content/ai-hw/scripts/export_anyscript_submission.py --mode extra_book --checkpoint {OUT}/best.pt --gallery_data_root {GALLERY_ROOT} --query_data_root {QUERY_ROOT} --out_csv {OUT}/submission_extra_book.csv --batch_size 4 --allow_synthetic_ids